In [2]:

import re
import html

file_path = "ASSIGNMENT ALL 3.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

# Clean HTML formatting without changing the original TXT file
text = html.unescape(text)
text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
text = re.sub(
    r"</?(?:em|strong)\b[^>\n]*>",
    "",
    text,
    flags=re.IGNORECASE
)
text = re.sub(r"<[^>\n]*>", "", text)

# Handle question 1 when it appears beside a session heading
text = re.sub(
    r"(?im)^([ \t]*SESSION[ \t]*\d+(?:[ \t]*\([^)]*\))?[ \t]*:[ \t]*)-[ \t]*(?=\d+\.)",
    r"\1\n",
    text
)

assignments = re.split(
    r"(?im)^[ \t]*FOLDER[ \t]*=[ \t]*",
    text
)
assignments = [a for a in assignments[1:] if a.strip()]

total_sessions = 0
total_questions = 0

for assignment in assignments:
    lines = assignment.strip().splitlines()
    folder_name = lines[0].strip()

    # Preserve the complete session label, including PART1/PART2
    sessions = list(re.finditer(
        r"(?im)^[ \t]*(SESSION[ \t]*\d+(?:[ \t]*\([^)]*\))?)[ \t]*:",
        assignment
    ))

    assignment_questions = 0

    print("\nASSIGNMENT:", folder_name)

    for i, session in enumerate(sessions):
        session_name = re.sub(
            r"\s+", " ", session.group(1).strip()
        )

        start = session.end()
        end = (
            sessions[i + 1].start()
            if i + 1 < len(sessions)
            else len(assignment)
        )

        section_text = assignment[start:end]

        questions = re.findall(
            r"(?ms)^[ \t]*(\d+)\.[ \t]*(.*?)(?=^[ \t]*\d+\.[ \t]*|\Z)",
            section_text
        )

        print(f"{session_name}: {len(questions)} questions")

        total_sessions += 1
        assignment_questions += len(questions)

    total_questions += assignment_questions
    print("Assignment total:", assignment_questions)

print("\n" + "=" * 40)
print("TOTAL ASSIGNMENTS:", len(assignments))
print("TOTAL SESSIONS:", total_sessions)
print("TOTAL QUESTIONS:", total_questions)



ASSIGNMENT: RETRIEVAL - AUGMENTED GEENRATION(RAG)
SESSION 1: 5 questions
SESSION 2: 5 questions
SESSION 3: 5 questions
SESSION 4: 5 questions
Assignment total: 20

ASSIGNMENT: PROMPT ENGINEERUNG
SESSION 1: 4 questions
SESSION 2: 5 questions
SESSION 3: 4 questions
SESSION 4: 5 questions
SESSION 5: 5 questions
Assignment total: 23

TOTAL ASSIGNMENTS: 2
TOTAL SESSIONS: 9
TOTAL QUESTIONS: 43


In [4]:
import os

# Parent folder where all assignment category folders will be created
output_path = r"F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS"

# Use the parent folder directly; category folders will be created from the TXT
assignment_path = output_path

os.makedirs(assignment_path, exist_ok=True)

print("Assignment output location is ready:")
print(assignment_path)

Assignment output location is ready:
F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS


In [5]:

import re
import html

# Read the new assignment file
with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

# Clean HTML formatting without changing the original TXT file
text = html.unescape(text)
text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
text = re.sub(
    r"</?(?:em|strong)\b[^>\n]*>",
    "",
    text,
    flags=re.IGNORECASE
)
text = re.sub(r"<[^>\n]*>", "", text)

# Fix question 1 when it appears beside a session heading
text = re.sub(
    r"(?im)^([ \t]*SESSION[ \t]*\d+(?:[ \t]*\([^)]*\))?[ \t]*:[ \t]*)-[ \t]*(?=\d+\.)",
    r"\1\n",
    text
)

# Extract assignments
assignments = re.split(
    r"(?im)^[ \t]*FOLDER[ \t]*=[ \t]*",
    text
)
assignments = [a.strip() for a in assignments[1:] if a.strip()]

# Extract sessions and questions, preserving the full session label
assignment_data = {}

for assignment in assignments:
    assignment_name = assignment.splitlines()[0].strip()

    sessions = list(re.finditer(
        r"(?im)^[ \t]*(SESSION[ \t]*\d+(?:[ \t]*\([^)]*\))?)[ \t]*:",
        assignment
    ))

    assignment_data[assignment_name] = {}

    for i, session in enumerate(sessions):
        session_name = re.sub(
            r"\s+", " ", session.group(1).strip()
        )

        start = session.end()
        end = (
            sessions[i + 1].start()
            if i + 1 < len(sessions)
            else len(assignment)
        )

        section_text = assignment[start:end]

        questions = re.findall(
            r"(?ms)^[ \t]*(\d+)\.[ \t]*(.*?)(?=^[ \t]*\d+\.[ \t]*|\Z)",
            section_text
        )

        assignment_data[assignment_name][session_name] = [
            question.strip()
            for number, question in questions
        ]

# Print totals
total_sessions = 0
total_questions = 0

for name, sections in assignment_data.items():
    print(f"\n{name}")

    for session_name, questions in sections.items():
        print(f"  {session_name}: {len(questions)} questions")
        total_sessions += 1
        total_questions += len(questions)

print("\nTotal sessions:", total_sessions)
print("Total questions:", total_questions)



RETRIEVAL - AUGMENTED GEENRATION(RAG)
  SESSION 1: 5 questions
  SESSION 2: 5 questions
  SESSION 3: 5 questions
  SESSION 4: 5 questions

PROMPT ENGINEERUNG
  SESSION 1: 4 questions
  SESSION 2: 5 questions
  SESSION 3: 4 questions
  SESSION 4: 5 questions
  SESSION 5: 5 questions

Total sessions: 9
Total questions: 43


In [6]:

import os
import json

# Create category folders and session notebooks from the TXT structure
created_notebooks = 0
skipped_notebooks = 0

for assignment_name, sessions in assignment_data.items():

    # Create the exact category folder name from the TXT file
    assignment_folder = os.path.join(
        output_path,
        assignment_name
    )
    os.makedirs(assignment_folder, exist_ok=True)

    for session_name, questions in sessions.items():

        # Create a clean session filename
        safe_session_name = session_name.replace(" ", "_")
        notebook_name = f"{safe_session_name}.ipynb"

        notebook_path = os.path.join(
            assignment_folder,
            notebook_name
        )

        # Never overwrite an existing notebook
        if os.path.exists(notebook_path):
            print("Already exists, skipped:", notebook_path)
            skipped_notebooks += 1
            continue

        cells = []

        # Notebook title
        cells.append({
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                f"# {assignment_name}\n",
                f"## {session_name}\n",
                "\n",
                "Assignment questions and answers\n"
            ]
        })

        # Add questions with answer placeholders
        for question_number, question in enumerate(questions, start=1):
            cells.append({
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    f"## Question {question_number}\n\n",
                    question,
                    "\n\n**Answer:**\n\n",
                    "Answer will be generated in the next step."
                ]
            })

        notebook = {
            "cells": cells,
            "metadata": {
                "kernelspec": {
                    "display_name": "Python 3",
                    "language": "python",
                    "name": "python3"
                }
            },
            "nbformat": 4,
            "nbformat_minor": 5
        }

        with open(notebook_path, "w", encoding="utf-8") as file:
            json.dump(notebook, file, indent=2, ensure_ascii=False)

        created_notebooks += 1
        print("Created:", notebook_path)

print("\nNew notebooks created:", created_notebooks)
print("Existing notebooks skipped:", skipped_notebooks)
print("Category folders processed:", len(assignment_data))
print("Total sessions detected:", sum(len(s) for s in assignment_data.values()))
print("Total questions detected:", sum(
    len(questions)
    for sessions in assignment_data.values()
    for questions in sessions.values()
))
print("Saved under:", output_path)


Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\RETRIEVAL - AUGMENTED GEENRATION(RAG)\SESSION_1.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\RETRIEVAL - AUGMENTED GEENRATION(RAG)\SESSION_2.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\RETRIEVAL - AUGMENTED GEENRATION(RAG)\SESSION_3.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\RETRIEVAL - AUGMENTED GEENRATION(RAG)\SESSION_4.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\PROMPT ENGINEERUNG\SESSION_1.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\PROMPT ENGINEERUNG\SESSION_2.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\PROMPT ENGINEERUNG\SESSION_3.ipynb
Created: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS\PROMPT ENGINEERUNG\SESSION_4.ipynb
Created: F:\DATA SCI

In [7]:

from google import genai
from google.genai import types

# Existing Google Cloud Vertex AI connection
client = genai.Client(
    enterprise=True,
    project="project-d24e3f00-1d87-4332-818",
    location="global",
    http_options=types.HttpOptions(timeout=120_000),
)

try:
    print("Testing Vertex AI Gemini 3.8 Flash with Interactions API...")

    response = client.interactions.create(
        model="gemini-3.8-flash",
        input="Reply with only the word SUCCESS.",
        generation_config={"thinking_level": "low"}
    )

    print("SUCCESS: Gemini responded.")
    print(response.output_text)

except Exception as e:
    print("ERROR:", type(e).__name__)
    print(str(e))


Testing Vertex AI Gemini 3.8 Flash with Interactions API...
SUCCESS: Gemini responded.
SUCCESS


In [ ]:

import json
import time
from pathlib import Path
from google import genai
from google.genai import types


# --------------------------------------------------
# 1. FOLDER AND VERTEX AI SETTINGS
# --------------------------------------------------

# Use only the category folders created from the current TXT file
ASSIGNMENT_FOLDERS = [
    Path(output_path) / assignment_name
    for assignment_name in assignment_data
]

for folder in ASSIGNMENT_FOLDERS:
    if not folder.exists():
        raise FileNotFoundError(
            f"Assignment folder not found: {folder}"
        )

PROJECT_ID = "project-d24e3f00-1d87-4332-818"
MODEL_NAME = "gemini-3.8-flash"
TIMEOUT_MS = 120_000

# Use Google Cloud Application Default Credentials (ADC).
# No Gemini API key or .env file is required.
client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location="global",
    http_options=types.HttpOptions(timeout=TIMEOUT_MS),
)

print(f"Vertex AI client ready. Model: {MODEL_NAME}")


# --------------------------------------------------
# 2. GENERATE AN ANSWER
# --------------------------------------------------



def generate_answer(question):
    prompt = f"""
You are an Agentic AI assignment tutor.

Answer the question below briefly and directly.
Follow exactly what the question asks.
Include only the essential points.
Use simple language.
Give Python code only if the question asks for code.
Avoid unnecessary explanations, introductions, conclusions, and extra examples.
For a definition, use 2-4 sentences.
For a list, provide only the required points with short explanations.
For a comparison, use a compact table when appropriate.

QUESTION:
{question}

Provide only the answer.
"""

    for attempt in range(1, 4):
        try:
            response = client.interactions.create(
                model=MODEL_NAME,
                input=prompt,
                generation_config={"thinking_level": "low"},
            )

            answer = response.output_text

            if not answer or not answer.strip():
                raise RuntimeError("Vertex AI returned an empty answer.")

            return answer.strip()

        except Exception as e:
            print(
                f"Request attempt {attempt}/3 failed: "
                f"{type(e).__name__}: {e}",
                flush=True,
            )

            if attempt == 3:
                raise

            wait_seconds = attempt * 5
            print(f"Retrying in {wait_seconds} seconds...", flush=True)
            time.sleep(wait_seconds)




# --------------------------------------------------
# 4. PROCESS THE SESSION NOTEBOOKS
# --------------------------------------------------

notebooks = sorted(
    [
        notebook
        for folder in ASSIGNMENT_FOLDERS
        for notebook in folder.glob("SESSION_*.ipynb")
    ],
    key=lambda p: (
        str(p.parent).lower(),
        int(re.search(r"SESSION_(\d+)", p.stem).group(1))
    )
)

print(f"Found {len(notebooks)} session notebooks.")

if not notebooks:
    raise RuntimeError("No SESSION_*.ipynb notebooks were found.")

new_answers = 0
skipped_answers = 0
MAX_NEW_ANSWERS = float("inf")

for notebook_path in notebooks:

    print(
    f"\nProcessing folder: {notebook_path.parent.name}"
    f" | Session: {notebook_path.name}",
    flush=True
)

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    notebook_changed = False

    for cell in notebook.get("cells", []):
        if new_answers >= MAX_NEW_ANSWERS:
            break

        if cell.get("cell_type") != "markdown":
            continue

        cell_text = "".join(cell.get("source", []))

        if "**Answer:**" not in cell_text:
            continue

        question_part, answer_part = cell_text.split(
            "**Answer:**", 1
        )

        existing_answer = answer_part.strip()

        # Skip answers that have already been generated.
        placeholder = "Answer will be generated in the next step."

        if existing_answer and existing_answer != placeholder:
            skipped_answers += 1
            continue

        # Extract the question after the question heading.
        question_parts = question_part.split("\n\n", 1)

        if len(question_parts) < 2:
            print("Skipping a cell because its question is missing.")
            continue

        question = question_parts[1].strip()
        question = question.removesuffix("\n")

        # Remove any leftover placeholder text from the question.
        if not question:
            print("Skipping an empty question.")
            continue

        print(
            f"Generating: {question[:100]}...",
            flush=True
        )

        try:
            answer = generate_answer(question)

        except Exception as e:
            print(f"\nSTOPPING GENERATOR: {e}", flush=True)
            print(
                "Previously saved answers remain on disk.",
                flush=True
            )
            raise

        # Update the answer in this markdown cell.
        updated_text = (
            question_part
            + "**Answer:**\n\n"
            + answer
        )

        cell["source"] = updated_text.splitlines(keepends=True)

        # Save immediately after each successful answer.
        with open(notebook_path, "w", encoding="utf-8") as f:
            json.dump(notebook, f, indent=2, ensure_ascii=False)

        new_answers += 1
        notebook_changed = True

        print(
            f"Saved answer #{new_answers} in {notebook_path.name}.",
            flush=True
        )

        time.sleep(1)

    if not notebook_changed:
        print("No new answers saved in this notebook.", flush=True)



# --------------------------------------------------
# 5. FINAL SUMMARY
# --------------------------------------------------

print("\n========== GENERATION SUMMARY ==========")
print(f"New answers saved: {new_answers}")
print(f"Existing answers skipped: {skipped_answers}")
print("Generator finished.")



Vertex AI client ready. Model: gemini-3.8-flash
Found 9 session notebooks.

Processing: SESSION_1.ipynb
Generating: Write two prompts for ChatGPT: one asking for a list of trending songs in India (unclear), and one a...
Saved answer #1 in SESSION_1.ipynb.
Generating: List 3 common mistakes people make when writing prompts for AI tools, and rewrite each mistake as a ...
Saved answer #2 in SESSION_1.ipynb.
Generating: Use ChatGPT or any AI chatbot to generate a restaurant review for a new café. First, give a vague pr...
Saved answer #3 in SESSION_1.ipynb.
Generating: Pick any feature from an app you use (e.g., Instagram's Explore page or Flipkart's product recommend...
Saved answer #4 in SESSION_1.ipynb.

Processing: SESSION_2.ipynb
Generating: Open ChatGPT and write a structured prompt asking for a step-by-step guide to create a new playlist ...
Saved answer #5 in SESSION_2.ipynb.
Generating: Write a prompt for ChatGPT that asks for a comparison of Zomato and Swiggy features, and instru

In [10]:

import json
from pathlib import Path

# Audit only the category folders detected from the current TXT file
folders = [
    Path(output_path) / assignment_name
    for assignment_name in assignment_data
]

placeholder = "Answer will be generated in the next step."

total = 0
completed = 0
pending = 0

print("=" * 85)
print("ASSIGNMENT BOT — SAVED ANSWER AUDIT")
print("=" * 85)

for folder in folders:
    print(f"\nCATEGORY: {folder.name}")

    for path in sorted(
        folder.glob("SESSION_*.ipynb"),
        key=lambda p: int(p.stem.split("_")[-1])
    ):
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        session_total = 0
        session_completed = 0
        session_pending = 0

        for cell in notebook.get("cells", []):
            if cell.get("cell_type") != "markdown":
                continue

            cell_text = "".join(cell.get("source", []))

            if "**Answer:**" not in cell_text:
                continue

            _, answer = cell_text.split("**Answer:**", 1)
            answer = answer.strip()

            session_total += 1

            if answer and answer != placeholder:
                session_completed += 1
            else:
                session_pending += 1

        total += session_total
        completed += session_completed
        pending += session_pending

        print(
            f"  {path.name:<22} "
            f"Saved: {session_completed:>3} | "
            f"Pending: {session_pending:>3} | "
            f"Total: {session_total:>3}"
        )

print("\n" + "-" * 85)
print(f"TOTAL ANSWERS: {total}")
print(f"SAVED ANSWERS: {completed}")
print(f"PENDING ANSWERS: {pending}")
print("=" * 85)


ASSIGNMENT BOT — SAVED ANSWER AUDIT

CATEGORY: RETRIEVAL - AUGMENTED GEENRATION(RAG)
  SESSION_1.ipynb        Saved:   5 | Pending:   0 | Total:   5
  SESSION_2.ipynb        Saved:   5 | Pending:   0 | Total:   5
  SESSION_3.ipynb        Saved:   5 | Pending:   0 | Total:   5
  SESSION_4.ipynb        Saved:   5 | Pending:   0 | Total:   5

CATEGORY: PROMPT ENGINEERUNG
  SESSION_1.ipynb        Saved:   4 | Pending:   0 | Total:   4
  SESSION_2.ipynb        Saved:   5 | Pending:   0 | Total:   5
  SESSION_3.ipynb        Saved:   4 | Pending:   0 | Total:   4
  SESSION_4.ipynb        Saved:   5 | Pending:   0 | Total:   5
  SESSION_5.ipynb        Saved:   5 | Pending:   0 | Total:   5

-------------------------------------------------------------------------------------
TOTAL ANSWERS: 43
SAVED ANSWERS: 43
PENDING ANSWERS: 0


In [9]:

import os

output_path = r"F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS"

print("Output folder:", output_path)
print("Folder exists:", os.path.exists(output_path))


Output folder: F:\DATA SCIENCE-AHEMDABAD\MACHINE LEARNING - DHRUMIL SIR\ASSIGNMENTS
Folder exists: True


In [11]:

import json
from pathlib import Path

folders = [
    Path(output_path) / assignment_name
    for assignment_name in assignment_data
]

placeholder = "Answer will be generated in the next step."
total = 0
completed = 0
pending = 0

for folder in folders:
    for path in sorted(
        folder.glob("SESSION_*.ipynb"),
        key=lambda p: int(p.stem.split("_")[-1])
    ):
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for cell in notebook.get("cells", []):
            if cell.get("cell_type") != "markdown":
                continue

            text = "".join(cell.get("source", []))
            if "**Answer:**" not in text:
                continue

            _, answer = text.split("**Answer:**", 1)
            answer = answer.strip()

            total += 1

            if not answer or placeholder in answer:
                pending += 1
            else:
                completed += 1

print("Total answer cells:", total)
print("Already completed:", completed)
print("Still pending:", pending)


Total answer cells: 43
Already completed: 43
Still pending: 0
